# BPE Tokenization Demo

This notebook demonstrates how Byte Pair Encoding (BPE) tokenization works, including some surprising behaviors.

In [10]:
# Install tiktoken if needed: pip install tiktoken

## 1. How BPE Actually Works: Iterative Adjacent Pair Merging

A common misconception is that BPE tries all possible tokenizations. It doesn't. BPE is a simple greedy algorithm:

1. Start with individual bytes
2. Find the **adjacent pair** with the lowest rank (most frequent in training)
3. Merge that pair
4. Repeat until no more merges are possible

The rank reflects the order merges were learned during training: lower rank = more frequent pattern.

In [11]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
ranks = enc._mergeable_ranks  # token -> rank mapping (lower = higher priority)

# Let's trace through "cann" step by step
text = "cann"
print(f"Input: '{text}'")
print(f"Actual tokenization: {[enc.decode([t]) for t in enc.encode(text)]}")
print()
print("Let's see why by tracing the algorithm...")

Input: 'cann'
Actual tokenization: ['c', 'ann']

Let's see why by tracing the algorithm...


In [12]:
print("Step 1: Start with bytes")
print("  ['c', 'a', 'n', 'n']")
print()
print("Step 2: Find lowest-ranked ADJACENT pair")
print("  Adjacent pairs:")
print(f"    'c'+'a' = 'ca' -> rank {ranks.get(b'ca', 'NOT IN VOCAB')}")
print(f"    'a'+'n' = 'an' -> rank {ranks[b'an']}")
print(f"    'n'+'n' = 'nn' -> rank {ranks[b'nn']}")
print("  Winner: 'an' (rank 276)")
print()
print("Step 3: Merge 'an'")
print("  ['c', 'an', 'n']")
print()
print("Step 4: Find lowest-ranked adjacent pair again")
print("  Adjacent pairs:")
print(f"    'c'+'an' = 'can' -> rank {ranks[b'can']}")
print(f"    'an'+'n' = 'ann' -> rank {ranks[b'ann']}")
print("  Winner: 'ann' (rank 1036)")
print()
print("Step 5: Merge 'ann'")
print("  ['c', 'ann']")
print()
print("Step 6: Check remaining pair")
print(f"    'c'+'ann' = 'cann' -> rank {ranks.get(b'cann', 'NOT IN VOCAB')}")
print("  No valid merge. Done!")
print()
print("Final result: ['c', 'ann']")

Step 1: Start with bytes
  ['c', 'a', 'n', 'n']

Step 2: Find lowest-ranked ADJACENT pair
  Adjacent pairs:
    'c'+'a' = 'ca' -> rank 936
    'a'+'n' = 'an' -> rank 276
    'n'+'n' = 'nn' -> rank 7521
  Winner: 'an' (rank 276)

Step 3: Merge 'an'
  ['c', 'an', 'n']

Step 4: Find lowest-ranked adjacent pair again
  Adjacent pairs:
    'c'+'an' = 'can' -> rank 4919
    'an'+'n' = 'ann' -> rank 1036
  Winner: 'ann' (rank 1036)

Step 5: Merge 'ann'
  ['c', 'ann']

Step 6: Check remaining pair
    'c'+'ann' = 'cann' -> rank NOT IN VOCAB
  No valid merge. Done!

Final result: ['c', 'ann']


**Key insight:** BPE only considers **adjacent pairs** at each step. It never tries all possible tokenizations.

Why `['c', 'ann']` instead of `['can', 'n']`?
- At step 2, 'an' (rank 276) gets merged first because it has the lowest rank
- This creates 'ann' as an adjacent pair in step 4
- 'ann' (rank 1036) then beats 'can' (rank 4919)

The order of merges matters! If 'ca' had a lower rank than 'an', we would have gotten a different result.

## 2. The Actual Encoding Algorithm

Here's the core of tiktoken's BPE implementation (from `src/lib.rs`):

```rust
fn _byte_pair_merge(ranks: &HashMap<Vec<u8>, Rank>, piece: &[u8]) -> Vec<(usize, Rank)> {
    // Start with individual bytes, track position and rank of each adjacent pair
    let mut parts = vec![];
    let mut min_rank: (Rank, usize) = (Rank::MAX, usize::MAX);
    
    // Find the lowest-ranked adjacent pair
    for i in 0..piece.len() - 1 {
        let rank = *ranks.get(&piece[i..i + 2]).unwrap_or(&Rank::MAX);
        if rank < min_rank.0 {
            min_rank = (rank, i);
        }
        parts.push((i, rank));
    }
    
    // Repeatedly merge lowest-ranked adjacent pair until done
    while min_rank.0 != Rank::MAX {
        let i = min_rank.1;
        // merge pair at position i, update affected neighbors
        // rescan for new minimum rank
    }
    parts
}
```

**Key points:**
- Only considers **adjacent pairs**, not all possible splits
- Greedy: always merge the lowest-ranked pair, then repeat
- O(n²) complexity where n is the input length

**Real implementations:**
- **tiktoken**: https://github.com/openai/tiktoken/blob/main/src/lib.rs
- **minbpe** (educational): https://github.com/karpathy/minbpe
- **SentencePiece**: https://github.com/google/sentencepiece